# Building End-to-End ETL Pipeline (API → Transform → PostgreSQL)


## Why This Matters

This is what real data engineers do:

- Pull data from APIs
- Clean & transform it
- Store it in databases

👉 Your first **production-shaped** pipeline


## Prerequisites

- PostgreSQL running ([`../../notebooks/02_sql/README.md`](../../notebooks/02_sql/README.md)).
- Network access for JSONPlaceholder.
- Logs write to `etl.log` next to this notebook (gitignored).

_If file logging seems stuck, **restart the kernel** and run from **Setup DB table** — `logging.basicConfig` only applies once per session._


## Setup DB table


In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="de_course",
    user="admin",
    password="admin",
)

cur = conn.cursor()

cur.execute(
    """
CREATE TABLE IF NOT EXISTS api_users (
    id INT PRIMARY KEY,
    name TEXT,
    email TEXT,
    city TEXT
)
"""
)

conn.commit()


## Extract (API)


In [ ]:
import requests


def extract():
    url = "https://jsonplaceholder.typicode.com/users"
    response = requests.get(url)
    return response.json()


## Transform (clean data)


In [ ]:
def transform(data):
    cleaned = []

    for d in data:
        try:
            cleaned.append(
                {
                    "id": d["id"],
                    "name": d["name"].strip().title(),
                    "email": d["email"].lower(),
                    "city": d["address"]["city"],
                }
            )
        except Exception:
            continue

    return cleaned


## Load (PostgreSQL)


In [ ]:
def load(data):
    for d in data:
        try:
            cur.execute(
                """
            INSERT INTO api_users (id, name, email, city)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (id) DO NOTHING
            """,
                (d["id"], d["name"], d["email"], d["city"]),
            )
        except Exception as e:
            print("Insert failed:", e)

    conn.commit()


## Run full pipeline


In [ ]:
def run_pipeline():
    raw = extract()
    cleaned = transform(raw)
    load(cleaned)

    print(f"Loaded {len(cleaned)} records")


run_pipeline()


## Validate data


In [ ]:
cur.execute("SELECT * FROM api_users LIMIT 5")
print(cur.fetchall())


## Add logging (production touch)

Writes to **`etl.log`** in the notebook working directory.


In [ ]:
import logging

logging.basicConfig(filename="etl.log", level=logging.INFO)


def run_pipeline_logged():
    logging.info("Pipeline started")

    raw = extract()
    cleaned = transform(raw)

    if not cleaned:
        logging.error("No data after transform")
        return

    load(cleaned)

    logging.info("Pipeline completed")


run_pipeline_logged()


## Practice

1. Map **`phone`** and **`company`** from JSONPlaceholder (`phone`, `company.name`).
2. Add columns + alter `INSERT` (or recreate table with migration mindset).
3. Query `SELECT ...` to verify.


## Assignment

Upgrade the pipeline:

- **Extract:** retries on failure
- **Logging:** structured messages (counts, duration)
- **Validate:** reject rows missing email or city

**Bonus:** `created_at TIMESTAMPTZ DEFAULT NOW()` on insert; time the run with `time.perf_counter()`.


## Interview Questions

1. What are ETL steps?
2. How do you handle duplicate inserts?
3. What is `ON CONFLICT`?
4. How do you validate data before loading?


## What you just built

- **API → clean → PostgreSQL**
- **Upsert semantics** via `ON CONFLICT DO NOTHING`
- **Logging** hook for operations

👉 Same shape as production pipelines—smaller scope.


---

**Next:** **Incremental pipelines** — process **new** data only; avoid full reloads.


## Optional: close connection


In [ ]:
cur.close()
conn.close()
print("Connection closed.")


## Your tasks

- [ ] Postgres up; run **Setup** → **Run full pipeline** → **Validate** (needs network).
- [ ] Run **Logging** cell; open `etl.log`; restart kernel if the file stays empty.
- [ ] **Practice:** add `phone` + company; migrate table / widen columns.
- [ ] **Assignment:** retries + validation + logging; **bonus:** timestamp column + run timing.
- [ ] Close connection when finished or leave open for the next notebook session.


## Shared dataset usage (`resources/datasets/`)

Use local reusable datasets for deterministic ETL tests:

```python
import json

with open("resources/datasets/users.json", encoding="utf-8") as f:
    users = json.load(f)

with open("resources/datasets/orders.json", encoding="utf-8") as f:
    orders = json.load(f)

print(len(users), len(orders))
```

For quality testing, switch input to `resources/datasets/dirty_users.json`.